In [2]:
!python -V
import psutil, platform, torch, sys
print("RAM (GB):", round(psutil.virtual_memory().total/1e9,2))
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())


Python 3.11.13
RAM (GB): 16.78
Platform: Linux-6.1.151+-x86_64-with-glibc2.35
CUDA available: False


In [4]:
PROJECT_ID = "instr-cs795-fall25-hqin-1"
BUCKET     = "instr-cs795-fall25-hqin-1-arasm002"
REGION = "us-central1"
EXPERIMENT = "tinyllm-phase2"

import os, time, pathlib
ts = time.strftime("%Y%m%d-%H%M%S")
ARTIFACTS = f"{BUCKET}/{EXPERIMENT}/{ts}"

print("PROJECT_ID:", PROJECT_ID)
print("REGION:", REGION)
print("ARTIFACTS:", ARTIFACTS)



PROJECT_ID: instr-cs795-fall25-hqin-1
REGION: us-central1
ARTIFACTS: instr-cs795-fall25-hqin-1-arasm002/tinyllm-phase2/20251023-021501


In [6]:
%%bash
pip -q install --upgrade pip
pip -q install transformers==4.43.4 accelerate==0.33.0 datasets==2.21.0 evaluate==0.4.2 optimum==1.21.4 torch --extra-index-url https://download.pytorch.org/whl/cpu scipy>=1.11.0 numpy>=1.24.0 psutil>=5.9.0 pandas>=2.2.0 matplotlib>=3.8.0 tqdm>=4.66.0
python - << 'PY'
import transformers, datasets, evaluate, pandas, psutil, torch, numpy
print("Installed OK.")
PY

Installed OK.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.1 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
2025-10-23 02:18:57.563505: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467]

In [9]:
import os, json, textwrap, pathlib
ROOT = "/content/tinyllm-phase2"
os.makedirs(ROOT, exist_ok=True)
for sub in ["scripts","models","data","results/charts"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)
print("Created:", ROOT)

# quantize_gptq.py
open(f"{ROOT}/scripts/quantize_gptq.py","w").write(r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-gptq-4bit")
    ap.add_argument("--bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    base = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map="auto")

    calib = get_calib_texts(1024)
    examples = [{"input_ids": tok(t, return_tensors="pt")["input_ids"]} for t in calib]

    qcfg = BaseQuantizeConfig(bits=args.bits, group_size=args.group_size, damp_percent=0.01, desc_act=True)
    qmodel = AutoGPTQForCausalLM.from_pretrained(base, quantize_config=qcfg)
    qmodel.quantize(examples)

    qmodel.save_pretrained(args.out)
    tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f:
        json.dump({"method": "gptq", "bits": args.bits, "group_size": args.group_size}, f, indent=2)
    print("[GPTQ] Saved to", args.out)

if __name__ == "__main__":
    main()
''')

# quantize_awq.py
open(f"{ROOT}/scripts/quantize_awq.py","w").write(r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM

def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-awq-4bit")
    ap.add_argument("--w_bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    model = AutoAWQForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map="auto")

    calib = get_calib_texts(1024)
    model.quantize(tokenizer=tok, calib_texts=calib, w_bits=args.w_bits, q_group_size=args.group_size, zero_point=True, version="GEMM")

    model.save_quantized(args.out, merge_lora=False, safetensors=True)
    tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f:
        json.dump({"method": "awq", "bits": args.w_bits, "group_size": args.group_size}, f, indent=2)
    print("[AWQ] Saved to", args.out)

if __name__ == "__main__":
    main()
''')

# evaluate.py
open(f"{ROOT}/scripts/evaluate.py","w").write(r'''#!/usr/bin/env python
import argparse, os, time, json, math, psutil, gc, pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline

RESULTS = Path("results")
DATA = Path("data")
RESULTS.mkdir(parents=True, exist_ok=True)

def ensure_eval_slices(seed=42):
    import random, json
    random.seed(seed)
    try:
        ds = load_dataset("hendrycks_test", "abstract_algebra", split="test")
        rows = []
        for ex in random.sample(list(ds), k=min(50, len(ds))):
            rows.append({"question": ex["question"], "choices": ex["choices"], "answer": ex["answer"]})
        with open(DATA/"eval_mmlu_50.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass
    try:
        ds = load_dataset("ai2_arc", "ARC-Easy", split="validation")
        rows = []
        for ex in random.sample(list(ds), k=min(50, len(ds))):
            rows.append({"question": ex["question"], "choices": ex["choices"]["text"], "answer": ex["answerKey"]})
        with open(DATA/"eval_arc_easy_50.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass
    try:
        ds = load_dataset("gsm8k", "main", split="test[:25]")
        rows = [{"question": r["question"], "answer": r["answer"]} for r in ds]
        with open(DATA/"eval_gsm8k_25.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass

def ppl_on_wikitext(model, tok, which="wikitext-2-raw-v1"):
    ds = load_dataset("wikitext", which, split="validation")
    enc = tok("\\n\\n".join(ds["text"]), return_tensors="pt")
    input_ids = enc["input_ids"]
    stride = 2048
    lls = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, input_ids.size(1), stride)):
            begin_loc = max(i + stride - 2048, 0)
            end_loc = min(i + stride, input_ids.size(1))
            trg_len = end_loc - i
            input_ids_slice = input_ids[:, begin_loc:end_loc]
            target_ids = input_ids_slice.clone()
            target_ids[:, :-trg_len] = -100
            out = model(input_ids_slice, labels=target_ids)
            lls.append(out.loss * trg_len)
    ppl = torch.exp(torch.stack(lls).sum() / end_loc).item()
    return ppl

def time_gen(pipe, prompt="Explain quantization in one paragraph.", gen_tokens=128):
    import time, psutil
    start_mem = psutil.Process().memory_info().rss
    t0 = time.time()
    _ = pipe(prompt, max_new_tokens=gen_tokens, do_sample=False)
    t1 = time.time()
    end_mem = psutil.Process().memory_info().rss
    total_time = t1 - t0
    tok_s = gen_tokens / total_time if total_time > 0 else float("nan")
    ms_per_tok = (total_time / gen_tokens) * 1000.0
    peak_ram_gb = max(start_mem, end_mem) / (1024**3)
    return {"latency_ms_per_token": ms_per_tok, "throughput_tok_s": tok_s, "peak_ram_gb": peak_ram_gb}

def load_model(which, model_name=None, model_dir=None):
    if which == "fp16":
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
        return model, tok
    elif which in ("gptq","awq"):
        tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype="auto", device_map="auto")
        return model, tok
    else:
        raise ValueError("which must be fp16|gptq|awq")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--which", required=True, choices=["fp16","gptq","awq"])
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--model_dir", default=None)
    ap.add_argument("--wiki", default="wikitext-2-raw-v1", choices=["wikitext-2-raw-v1","wikitext-103-raw-v1"])
    args = ap.parse_args()

    ensure_eval_slices()

    model, tok = load_model(args.which, model_name=args.model, model_dir=args.model_dir)

    ppl = ppl_on_wikitext(model, tok, args.wiki)
    pipe = TextGenerationPipeline(model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)
    perf = time_gen(pipe)

    row = {
        "method": args.which,
        "model_dir_or_name": args.model if args.which=="fp16" else args.model_dir,
        "ppl": ppl,
        "peak_ram_gb": perf["peak_ram_gb"],
        "throughput_tok_s": perf["throughput_tok_s"],
        "latency_ms_per_token": perf["latency_ms_per_token"],
    }
    import pandas as pd, os
    os.makedirs("results", exist_ok=True)
    csv_path = "results/metrics.csv"
    df = pd.DataFrame([row])
    if os.path.exists(csv_path):
        old = pd.read_csv(csv_path)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print("[Eval] Wrote", csv_path)

    # simple plots
    import matplotlib.pyplot as plt
    full = pd.read_csv(csv_path)
    os.makedirs("results/charts", exist_ok=True)
    plt.figure()
    full.groupby("method")["ppl"].mean().plot(kind="bar", title="PPL (lower is better)")
    plt.tight_layout(); plt.savefig("results/charts/accuracy_vs_size.png"); plt.close()

    plt.figure()
    full.groupby("method")["throughput_tok_s"].mean().plot(kind="bar", title="Throughput (tok/s)")
    plt.tight_layout(); plt.savefig("results/charts/speed_vs_ram.png"); plt.close()
    print("[Eval] Charts saved in results/charts")

if __name__ == "__main__":
    main()
''')

# make_plots.py
open(f"{ROOT}/scripts/make_plots.py","w").write(r'''#!/usr/bin/env python
import pandas as pd, matplotlib.pyplot as plt, os
df = pd.read_csv("results/metrics.csv")
os.makedirs("results/charts", exist_ok=True)
plt.figure(); df.groupby("method")["ppl"].mean().plot(kind="bar", title="PPL (lower is better)"); plt.tight_layout(); plt.savefig("results/charts/accuracy_vs_size.png"); plt.close()
plt.figure(); df.groupby("method")["throughput_tok_s"].mean().plot(kind="bar", title="Throughput (tok/s)"); plt.tight_layout(); plt.savefig("results/charts/speed_vs_ram.png"); plt.close()
print("Saved charts to results/charts/")
''')


Created: /content/tinyllm-phase2


568

In [12]:
# export_to_gguf.py (fixed quoting)
open(f"{ROOT}/scripts/export_to_gguf.py","w").write(r'''#!/usr/bin/env python
HELP = """Steps:
1) git clone https://github.com/ggerganov/llama.cpp
2) cd llama.cpp && make -j
3) python ./convert-hf-to-gguf.py --model <HF_ID_OR_DIR> --outfile <OUT.gguf> --outtype f16
4) ./quantize <OUT.gguf> <OUT-q4_0.gguf> q4_0
5) ./main -m <OUT-q4_0.gguf> -p 'Explain quantization in 2 sentences.' -n 128 --threads $(nproc)
"""

def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--hf_model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out_gguf", default="/content/tinyllm-phase2/models/phi3mini-f16.gguf")
    ap.add_argument("--llama_dir", default="/content/llama.cpp")
    args = ap.parse_args()
    print(HELP)
    print("Suggested command:")
    print(f"python {args.llama_dir}/convert-hf-to-gguf.py --model {args.hf_model} "
          f"--outfile {args.out_gguf} --outtype f16")

if __name__ == "__main__":
    main()
''')
print("Scripts written.")

Scripts written.


In [16]:
%%bash
set -e

# Base tooling
pip -q install -U pip setuptools wheel

# Stay on a solid torch for CPU, compatible with auto-gptq & transformers
pip -q install --upgrade --extra-index-url https://download.pytorch.org/whl/cpu \
  "torch==2.2.2+cpu" "torchvision==0.17.2+cpu" "torchaudio==2.2.2+cpu"

# Core ML/NLP stack (pinned)
pip -q install "transformers==4.43.4" "accelerate==0.33.0" \
               "datasets==2.21.0" "evaluate==0.4.2" "optimum==1.21.4" \
               "safetensors>=0.4.0" "sentencepiece>=0.1.99" "tqdm>=4.66.0"

# Resolve common preinstalled image conflicts
pip -q install "protobuf==4.25.3" "numpy==2.0.2" "pandas>=2.2.0" "psutil>=5.9.0" "matplotlib>=3.8.0" "scipy>=1.11.0"

# GPTQ
pip -q install "auto-gptq==0.7.1"

# AWQ — use the maintained PyPI package name: autoawq (import path is 'awq')
pip -q install "autoawq==0.2.6.post2" || pip -q install "autoawq==0.2.6"

python - << 'PY'
import torch
print("torch:", torch.__version__, "cuda?", torch.cuda.is_available())
try:
    import auto_gptq
    print("auto_gptq:", getattr(auto_gptq, "__version__", "imported"))
except Exception as e:
    print("auto_gptq import ERROR:", e)
try:
    import awq
    from awq import AutoAWQForCausalLM
    print("awq: imported; AutoAWQForCausalLM available")
except Exception as e:
    print("awq import ERROR:", e)
PY


torch: 2.3.1+cu121 cuda? False
auto_gptq import ERROR: cannot import name 'HybridCache' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)
awq: imported; AutoAWQForCausalLM available


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 0.33.0 requires numpy<2.0.0,>=1.17, but you have numpy 2.0.2 which is incompatible.
optimum 1.21.4 requires numpy<2.0, but you have numpy 2.0.2 which is incompatible.
grpcio-status 1.75.1 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 4.25.3 which is incompatible.
google-colabsqlviz 0.2.5 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 4.25.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but 

In [18]:
%%bash
set -e

# Clean, compatible stack for CPU
pip -q install -U pip setuptools wheel

# Torch CPU (stable with auto-gptq & autoawq)
pip -q install --upgrade --extra-index-url https://download.pytorch.org/whl/cpu \
  "torch==2.2.2+cpu" "torchvision==0.17.2+cpu" "torchaudio==2.2.2+cpu"

# Core libs – pin to versions that include HybridCache and match PEFT
pip -q install "transformers==4.46.1" "accelerate==0.33.0" \
               "datasets==2.21.0" "evaluate==0.4.2" "optimum==1.21.4" \
               "peft==0.13.2" "safetensors>=0.4.0" "sentencepiece>=0.1.99" \
               "tqdm>=4.66.0"

# Resolve common preinstalled conflicts
pip -q install "protobuf==4.25.3" "numpy==2.0.2" "pandas>=2.2.0" "psutil>=5.9.0" "matplotlib>=3.8.0" "scipy>=1.11.0"

# Quant tools
pip -q install "auto-gptq==0.7.1"         # GPTQ
pip -q install "autoawq==0.2.6.post2" || pip -q install "autoawq==0.2.6"   # AWQ

python - << 'PY'
import torch, transformers, peft
print("torch:", torch.__version__, "cuda?", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
# Smoke imports
import auto_gptq
from awq import AutoAWQForCausalLM
print("auto_gptq OK, awq OK")
PY


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autoawq-kernels 0.0.7 requires torch==2.3.1, but you have torch 2.2.2+cpu which is incompatible.
autoawq 0.2.6 requires torch==2.3.1, but you have torch 2.2.2+cpu which is incompatible.
ERROR: Cannot install optimum==1.21.4 and transformers==4.46.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


CalledProcessError: Command 'b'set -e\n\n# Clean, compatible stack for CPU\npip -q install -U pip setuptools wheel\n\n# Torch CPU (stable with auto-gptq & autoawq)\npip -q install --upgrade --extra-index-url https://download.pytorch.org/whl/cpu \\\n  "torch==2.2.2+cpu" "torchvision==0.17.2+cpu" "torchaudio==2.2.2+cpu"\n\n# Core libs \xe2\x80\x93 pin to versions that include HybridCache and match PEFT\npip -q install "transformers==4.46.1" "accelerate==0.33.0" \\\n               "datasets==2.21.0" "evaluate==0.4.2" "optimum==1.21.4" \\\n               "peft==0.13.2" "safetensors>=0.4.0" "sentencepiece>=0.1.99" \\\n               "tqdm>=4.66.0"\n\n# Resolve common preinstalled conflicts\npip -q install "protobuf==4.25.3" "numpy==2.0.2" "pandas>=2.2.0" "psutil>=5.9.0" "matplotlib>=3.8.0" "scipy>=1.11.0"\n\n# Quant tools\npip -q install "auto-gptq==0.7.1"         # GPTQ\npip -q install "autoawq==0.2.6.post2" || pip -q install "autoawq==0.2.6"   # AWQ\n\npython - << \'PY\'\nimport torch, tr

In [17]:
%cd /content/tinyllm-phase2
!python scripts/quantize_gptq.py --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-gptq-4bit --bits 4 --group_size 128
!python scripts/quantize_awq.py  --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-awq-4bit  --w_bits 4 --group_size 128


/content/tinyllm-phase2
Traceback (most recent call last):
  File "/content/tinyllm-phase2/scripts/quantize_gptq.py", line 5, in <module>
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
  File "/usr/local/lib/python3.11/dist-packages/auto_gptq/__init__.py", line 3, in <module>
    from .utils.peft_utils import get_gptq_peft_model
  File "/usr/local/lib/python3.11/dist-packages/auto_gptq/utils/peft_utils.py", line 6, in <module>
    from peft import PeftConfig, PeftModel, PeftType, get_peft_model
  File "/usr/local/lib/python3.11/dist-packages/peft/__init__.py", line 17, in <module>
    from .auto import (
  File "/usr/local/lib/python3.11/dist-packages/peft/auto.py", line 32, in <module>
    from .peft_model import (
  File "/usr/local/lib/python3.11/dist-packages/peft/peft_model.py", line 37, in <module>
    from transformers import Cache, DynamicCache, EncoderDecoderCache, HybridCache, PreTrainedModel
ImportError: cannot import name 'HybridCache' from 'transformers'